**Programmer: python_scripts (Abhijith Warrier)**

**PYTHON SCRIPT TO *TURN A PRODUCTION DECISION INTO AN OPTIMIZATION PROBLEM WITH AN OBJECTIVE AND RESOURCE CONSTRAINTS. 🐍🎯***

A workshop makes standard and premium bags. Premium bags earn more per item, but they also require more time. Which mix should the workshop make with limited resources?

Optimization separates the decisions we control, the objective we want to improve, and the constraints every valid plan must satisfy. We will formulate a small integer model, enumerate all bounded choices, and verify the best feasible plan. All values are fictional teaching data, not business measurements.

---

## 1. Requirements
Python 3.9+ and Jupyter. The example uses only the standard library; no additional packages are needed.

---

## 2. Imports
`product` generates every pair of candidate quantities.

In [1]:
from itertools import product

---

## 3. Define the decisions and assumptions
Let **x** be standard bags and **y** be premium bags. Both are nonnegative integers.

| Per bag | Standard | Premium |
|---|---:|---:|
| Profit contribution (units) | 30 | 40 |
| Time (hours) | 2 | 4 |
| Fabric (units) | 3 | 2 |

Available: **16 hours** and **18 fabric units**. Assume each bag sells, unit contributions and resource use are fixed, and there are no setup costs or other limits. Profit units are illustrative contribution after variable costs, before any fixed costs.

In [2]:
profit_standard, profit_premium = 30, 40
hours_standard, hours_premium = 2, 4
fabric_standard, fabric_premium = 3, 2
hours_available, fabric_available = 16, 18

# Upper bounds come from making only one product at a time.
max_x = min(hours_available // hours_standard,
            fabric_available // fabric_standard)
max_y = min(hours_available // hours_premium,
            fabric_available // fabric_premium)
print("Search bounds:", f"0 <= x <= {max_x}, 0 <= y <= {max_y}")
assert (max_x, max_y) == (6, 4)

Search bounds: 0 <= x <= 6, 0 <= y <= 4


---

## 4. Write the objective and constraints
**Maximize:** `30x + 40y`.

**Subject to:**
- `2x + 4y <= 16` (hours)
- `3x + 2y <= 18` (fabric)
- `x, y` are nonnegative integers.

The objective ranks acceptable plans. Constraints decide which plans are acceptable.

In [3]:
def evaluate(x, y):
    hours = hours_standard * x + hours_premium * y
    fabric = fabric_standard * x + fabric_premium * y
    return {
        "standard": x, "premium": y,
        "profit": profit_standard * x + profit_premium * y,
        "hours": hours, "fabric": fabric,
        "feasible": (isinstance(x, int) and isinstance(y, int)
                     and x >= 0 and y >= 0
                     and hours <= hours_available
                     and fabric <= fabric_available),
    }

for x, y in [(0, 4), (4, 2), (6, 4)]:
    print(evaluate(x, y))

{'standard': 0, 'premium': 4, 'profit': 160, 'hours': 16, 'fabric': 8, 'feasible': True}
{'standard': 4, 'premium': 2, 'profit': 200, 'hours': 16, 'fabric': 16, 'feasible': True}
{'standard': 6, 'premium': 4, 'profit': 340, 'hours': 28, 'fabric': 26, 'feasible': False}


---

## 5. Search feasible plans
The derived bounds include every feasible nonnegative integer plan: neither product can exceed its individual resource limit. This tiny problem needs only 35 candidate checks. Exhaustive enumeration establishes the global optimum here, but scales poorly for larger models.

In [4]:
plans = [evaluate(x, y)
         for x, y in product(range(max_x + 1), range(max_y + 1))]
feasible = [plan for plan in plans if plan["feasible"]]
if not feasible:
    raise ValueError("No feasible production plan.")

best_profit = max(plan["profit"] for plan in feasible)
# Retain all ties, rather than hiding alternative optimal plans.
best_plans = [plan for plan in feasible if plan["profit"] == best_profit]
print("Candidates:", len(plans), "| Feasible:", len(feasible))
print("Best plans:", best_plans)
assert len(plans) == 35 and len(feasible) == 22
assert [(p["standard"], p["premium"]) for p in best_plans] == [(4, 2)]
assert best_profit == 200

Candidates: 35 | Feasible: 22
Best plans: [{'standard': 4, 'premium': 2, 'profit': 200, 'hours': 16, 'fabric': 16, 'feasible': True}]


---

## 6. Verify and interpret
The best plan uses all available hours, but leaves fabric unused. A binding constraint has no remaining slack; here time is binding and fabric is not. Using every resource is not the objective.

In [5]:
best = best_plans[0]
assert best["hours"] == 2 * 4 + 4 * 2 == 16
assert best["fabric"] == 3 * 4 + 2 * 2 == 16
assert best["profit"] == 30 * 4 + 40 * 2 == 200
assert all(plan["profit"] <= best["profit"] for plan in feasible)
print("Standard bags:", best["standard"])
print("Premium bags:", best["premium"])
print("Profit units:", best["profit"])
print("Unused hours:", hours_available - best["hours"])
print("Unused fabric:", fabric_available - best["fabric"])
print("All-premium feasible plan profit:", evaluate(0, 4)["profit"])

Standard bags: 4
Premium bags: 2
Profit units: 200
Unused hours: 0
Unused fabric: 2
All-premium feasible plan profit: 160


---

## Takeaway
Define **variables → objective → constraints → solution → verification**. This is a small integer linear optimization problem solved by exhaustive enumeration, not by an external solver. “Best” is relative to the modeled objective and assumptions. Demand limits, uncertain sales, setup costs, or changing resource use would require revising the model.

---